# Deep Learning

> [Kaggle Course - Intro to Deep Learning](https://www.kaggle.com/learn/intro-to-deep-learning)

**Lessons**
1. [A Single Neuron](https://www.kaggle.com/code/ryanholbrook/a-single-neuron)
    - Linear Unit or **Neuron**
    - Input Shape

2. [Deep Neural Networks](https://www.kaggle.com/code/ryanholbrook/deep-neural-networks)
    - Layers & Dense
    - Activation Functions: Rectified Linear Unit (ReLU), ELU, Sigmoid, etc.

3. [Stochastic Gradient Descent](https://www.kaggle.com/code/ryanholbrook/stochastic-gradient-descent)
    - Loss Functions: Tells what problem to solve (or) simply the objective.
        - Examples: Mean Squared Error (MSE), Mean Absolute Error (MAE), etc.
    - Optimizers: How to solve the problem. Example: Change the weights to minimize the loss.
        - MiniBatch
        - Epoch

    | Metric    | Values                                                            |
    |-----------|-------------------------------------------------------------------|
    | Dataset   | 1000 training rows / examples                                     |
    | MiniBatch | 100 - Each contains 10 rows / examples                            |
    | 1 Epoch   | 10 MiniBatches (i.e, one round trip of entrie datasets)           |
    | 10 Epoch  | 10 x 10 = 100 MiniBatches (i.e, 10 round trip of entrie datasets) |

4. [Overfitting and Underfitting]()

5. [Dropout and Batch Normalization]()

6. [Binary Classification]()

7. [Bonus: Detecting the Higgs Boson with TPUs]()

## Lesson 1 - A Single Neuron
- Learn about linear units, the building blocks of deep learning

#### Step 1 - Import necessary frameworks
- tensorflow, pandas

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import pandas as pd

#### Step 2 - Load the data

In [ ]:
red_wine_data = pd.read_csv("../../resources/red-wine.csv")

#### Step 3 - Inspect Data

In [ ]:
print(f"Shape: {red_wine_data.shape}")
print(f"Columns:\n{red_wine_data.columns}")
print(f"First 5 rows:\n{red_wine_data.head()}")

In [ ]:
red_wine_data.columns[0: -1]

#### Step 4 - Define Model

| Metric | Value                                                                                                                                                                             |
|------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Inputs | 11 - ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides',  'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol'] |
| Output | quality                                                                                                                                                                           |
| Library | tensorflow.keras                                                                                                                                                                  |
| input_shape | (11, )                                                                                                                                                                            |

In [ ]:
input_shape = (11, )
single_neuron_layer = layers.Dense(units = 1, input_shape = input_shape)
model = keras.Sequential([single_neuron_layer])
model.summary()

#### Step 5 - Inspect Model

In [ ]:
weights, bias = model.weights
print(f"Input Shape : {model.input_shape}, Output Shape : {model.output_shape}")
print(f"Weights: {weights}")
print(f"Bias: {bias}")

## Lesson 2 - Deep Neural Networks

#### Step 1 - Multi Layer Neural Network

In [ ]:
concrete_data = pd.read_csv('../../resources/concrete.csv')
concrete_data.head()

#### Step 2 - Define the Model

| Metric                       | Value                                                                                              |
|------------------------------|----------------------------------------------------------------------------------------------------|
| Inputs                       | 8 - Cement, BlastFurnaceSlag, FlyAsh, Water, Superplasticizer, CoarseAggregate, FineAggregate, Age |
| Output                       | CompressiveStrength                                                                                |
| Hidden Layers & Neurons/Unit | 3 & 512 units                                                                                      |
| Hidden Layer Activation      | ReLU (Rectified Linear Unit)                                                                       |
| Output Layer                 | 1                                                                                                  |

In [ ]:
input_columns = concrete_data.columns[0: -1]
output_column = concrete_data.columns[-1]

print(f"Input: {input_columns} | Total : {len(input_columns)}\nOutput: {output_column}")

In [ ]:
concrete_model = keras.Sequential(
    [
        # 3 Hidden Layers with 512 neurons each
        layers.Dense(units = 512, activation = 'relu', input_shape = (8, )),
        layers.Dense(units = 512, activation = 'relu'),
        layers.Dense(units = 512, activation = 'relu'),

        # Output Layer
        layers.Dense(1),
    ]
)

#### Step 3 - Explore the Model

In [ ]:
weights = concrete_model.get_weights()
for w in weights:
    print(w.shape)

#### Step 4 - Neural Network Architecture For a Single Row

![image](../../resources/images/lesson_2_neural_network_arch.png)

## Lesson 3 - Stochastic Gradient Descent

#### Import the necessary frameworks
    - numpy, pandas, sklearn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

#### Step 1 - Load the fuel data

In [ ]:
fuel_data = pd.read_csv("../../resources/fuel.csv")
fuel_data.head()

#### Step 2 - Understand the data

1. Target - Derive Fuel Economy 'FE' as output
2. Features - Remaining columns excluding the target 'FE'

In [ ]:
# make a deep copy of the original fuel data
X = fuel_data.copy()

# target column
y = X.pop('FE')

print(y.head())
print(X.head())

In [ ]:
preprocessor = make_column_transformer(
    # SS normalize the numerical features / columns
    (StandardScaler(), make_column_selector(dtype_include = np.number)),

    # OHE converts categorical features to numerical features, and bloats up the columns
    (OneHotEncoder(sparse_output=False), make_column_selector(dtype_include = object))
)

X = preprocessor.fit_transform(X)
y = np.log(y)


In [ ]:
print(X.shape)
print(y.shape)
print((X.shape[1], ))

In [ ]:
pp_features = preprocessor.get_feature_names_out()
for name in pp_features:
    print(name)

#### Step 3 - Define the Model

In [ ]:
fuel_model = keras.Sequential(
    [
        layers.Input(shape = (X.shape[1], )),
        layers.Dense(units = 128, activation = 'relu'),
        layers.Dense(units = 128, activation = 'relu'),
        layers.Dense(units = 64, activation = 'relu'),
        layers.Dense(units = 1)
    ]
)

#### Step 5 - Add Loss and Optimizer

In [ ]:
fuel_model.compile(optimizer="Adam", loss="mae")

#### Step 6 - Train the Model

In [ ]:
fuel_training_history = fuel_model.fit(x=X, y=y, batch_size=128, epochs=200)

#### Step 7 - Inspect Loss from training

In [ ]:
fuel_training_history_df = pd.DataFrame(fuel_training_history.history)
fuel_training_history_df.loc[5:, ['loss']].plot()

## Lesson 4 - Overfitting and Underfitting

In [ ]:
from IPython.display import display

df_red_wine_train = red_wine_data.sample(frac=0.7, random_state=0)
df_red_wine_valid = red_wine_data.drop(df_red_wine_train.index)
display(df_red_wine_train.head())

In [ ]:
max_red_wine = df_red_wine_train.max(axis=0)
min_red_wine = df_red_wine_train.min(axis=0)
df_red_wine_train = (df_red_wine_train - min_red_wine) / (max_red_wine - min_red_wine)
df_red_wine_valid = (df_red_wine_valid - min_red_wine) / (max_red_wine - min_red_wine)

X_rw_train = df_red_wine_train.drop('quality', axis=1)
X_rw_valid = df_red_wine_valid.drop('quality', axis=1)

y_rw_train = df_red_wine_train['quality']
y_rw_valid = df_red_wine_valid['quality']

#### Step 1 - Add Early Stopping callback

In [ ]:
early_stopping = callbacks.EarlyStopping(
    min_delta=0.001,
    patience=20,
    restore_best_weights=True
)

red_wine_model = keras.Sequential(
    [
        layers.Input(shape=(11, )),
        layers.Dense(units=512, activation='relu'),
        layers.Dense(units=512, activation='relu'),
        layers.Dense(units=512, activation='relu'),
        layers.Dense(units=1)
    ]
)

red_wine_model.compile(
    optimizer="Adam",
    loss="mae"
)

In [ ]:
red_wine_history = red_wine_model.fit(
    X_rw_train,
    y_rw_train,
    batch_size=256,
    epochs=500,
    validation_data=(X_rw_valid, y_rw_valid),
    callbacks=[early_stopping],
    verbose=0
)

red_wine_history_df = pd.DataFrame(red_wine_history.history)
red_wine_history_df.loc[:, ['loss', 'val_loss']].plot()

In [ ]:
print("Minimum validation loss: {}".format(red_wine_history_df['val_loss'].min()))

## Lesson 5 - Dropout and Batch Normalization

In [ ]:
red_wine_model = keras.Sequential(
    [
        layers.Input(shape=(11, )),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.Dense(units=1024, activation='relu'),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.Dense(units=1024, activation='relu'),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.Dense(units=1),
    ]
)

red_wine_model.compile(
    optimizer="Adam",
    loss="mae"
)

In [ ]:
red_wine_history = red_wine_model.fit(
    X_rw_train,
    y_rw_train,
    validation_data=(X_rw_valid, y_rw_valid),
    batch_size=256,
    epochs=100,
    verbose=0
)

red_wine_history_df = pd.DataFrame(red_wine_history.history)
red_wine_history_df.loc[:, ['loss', 'val_loss']].plot()